In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


PROJECT = Path(r"Z:\Projects\monsoon-postprocessing")

PREDICTOR_FOLDER = (
    PROJECT
    / "data"
    / "raw"
    / "gefs_predictors"
)

RAINFALL_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_gefs_imerg.nc"
)

DAILY_FOLDER = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_predictor_daily"
)

PREDICTOR_OUTPUT = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_predictors.nc"
)

MODEL_DATASET_OUTPUT = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_model_dataset.nc"
)

DAILY_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

In [2]:
with xr.open_dataset(RAINFALL_FILE) as ds:
    rainfall_ds = ds.load()

target_latitude = rainfall_ds.latitude
target_longitude = rainfall_ds.longitude

print("Target latitude cells:", target_latitude.size)
print("Target longitude cells:", target_longitude.size)
print("Dates:", rainfall_ds.sizes["date"])

Target latitude cells: 129
Target longitude cells: 121
Dates: 31


In [3]:
def process_predictor_file(
    file,
    predictor_name
):
    """Open, crop and interpolate one predictor field."""

    with xr.open_dataset(
        file,
        engine="cfgrib",
        backend_kwargs={
            "indexpath": ""
        }
    ) as ds:
        variable_name = list(ds.data_vars)[0]
        field = ds[variable_name].load()

    field = field.squeeze(drop=True)

    field = field.sortby("latitude")
    field = field.sortby("longitude")

    # Keep a small buffer for boundary interpolation.
    field = field.where(
        (field.latitude >= 5.5)
        & (field.latitude <= 38.5)
        & (field.longitude >= 67.5)
        & (field.longitude <= 98.5),
        drop=True
    )

    field = field.interp(
        latitude=target_latitude,
        longitude=target_longitude,
        method="linear"
    )

    values = field.values.astype(
        np.float32
    )

    original_units = str(
        field.attrs.get("units", "")
    )

    if predictor_name == "mslp":
        # Convert pressure from Pa to hPa.
        if np.nanmedian(values) > 2000:
            values = values / 100

        units = "hPa"

    elif predictor_name == "q850":
        # Convert specific humidity from kg/kg to g/kg.
        if np.nanmedian(values) < 1:
            values = values * 1000

        units = "g/kg"

    else:
        units = "m/s"

    cleaned = xr.DataArray(
        values,
        coords={
            "latitude": target_latitude,
            "longitude": target_longitude
        },
        dims=(
            "latitude",
            "longitude"
        ),
        name=predictor_name
    )

    cleaned.attrs = {
        "units": units,
        "original_units": original_units,
        "forecast_hour": 36
    }

    return cleaned

In [4]:
test_file = (
    PREDICTOR_FOLDER
    / "mslp_2018070100_f036.grib2"
)

test_mslp = process_predictor_file(
    test_file,
    "mslp"
)

print(test_mslp)

print(
    "MSLP range:",
    float(test_mslp.min(skipna=True)),
    "to",
    float(test_mslp.max(skipna=True)),
    "hPa"
)

<xarray.DataArray 'mslp' (latitude: 129, longitude: 121)> Size: 62kB
array([[1008.1793 , 1008.1353 , 1008.1453 , ..., 1009.0413 , 1009.1453 ,
        1009.1693 ],
       [1008.20734, 1008.15735, 1008.1293 , ..., 1009.06134, 1009.1493 ,
        1009.17737],
       [1008.15326, 1008.1253 , 1008.1133 , ..., 1009.0253 , 1009.0574 ,
        1009.08325],
       ...,
       [ 997.18134,  997.25134,  996.9693 , ..., 1002.01733, 1002.3773 ,
        1002.9813 ],
       [ 997.7693 ,  998.0133 ,  997.6133 , ..., 1003.6753 , 1004.0453 ,
        1004.12134],
       [ 997.6053 ,  998.3393 ,  998.3033 , ..., 1006.06134, 1006.2573 ,
        1006.0633 ]], shape=(129, 121), dtype=float32)
Coordinates:
  * latitude   (latitude) float64 1kB 6.0 6.25 6.5 6.75 ... 37.5 37.75 38.0
  * longitude  (longitude) float64 968B 68.0 68.25 68.5 ... 97.5 97.75 98.0
Attributes:
    units:           hPa
    original_units:  Pa
    forecast_hour:   36
MSLP range: 981.8673706054688 to 1013.2713012695312 hPa


In [5]:
observation_dates = pd.date_range(
    start="2018-07-01",
    end="2018-07-31",
    freq="D"
)

predictor_names = [
    "mslp",
    "u850",
    "v850",
    "q850"
]

processing_rows = []


for observation_date in observation_dates:
    initialization_date = (
        observation_date
        - pd.Timedelta(days=1)
    )

    initialization = initialization_date.strftime(
        "%Y%m%d00"
    )

    daily_file = (
        DAILY_FOLDER
        / f"predictors_{observation_date:%Y%m%d}.nc"
    )

    if daily_file.exists() and daily_file.stat().st_size > 0:
        print(
            "Already processed:",
            observation_date.date()
        )

        processing_rows.append({
            "date": observation_date.date(),
            "success": True
        })

        continue

    try:
        daily_variables = {}

        for predictor_name in predictor_names:
            predictor_file = (
                PREDICTOR_FOLDER
                / (
                    f"{predictor_name}_"
                    f"{initialization}_f036.grib2"
                )
            )

            if not predictor_file.exists():
                raise FileNotFoundError(
                    predictor_file
                )

            daily_variables[predictor_name] = (
                process_predictor_file(
                    predictor_file,
                    predictor_name
                )
            )

        daily_ds = xr.Dataset(
            daily_variables
        )

        daily_ds["wind_speed_850"] = np.sqrt(
            daily_ds["u850"] ** 2
            + daily_ds["v850"] ** 2
        )

        daily_ds[
            "wind_speed_850"
        ].attrs["units"] = "m/s"

        daily_ds = daily_ds.expand_dims(
            date=[
                observation_date.to_datetime64()
            ]
        )

        daily_ds.attrs = {
            "observation_date": str(
                observation_date.date()
            ),
            "gefs_initialization": str(
                initialization_date.date()
            ),
            "forecast_hour": 36
        }

        temporary_file = daily_file.with_suffix(
            ".tmp.nc"
        )

        daily_ds.to_netcdf(
            temporary_file,
            engine="netcdf4"
        )

        temporary_file.replace(
            daily_file
        )

        processing_rows.append({
            "date": observation_date.date(),
            "success": True
        })

        print(
            "Processed:",
            observation_date.date()
        )

    except Exception as error:
        processing_rows.append({
            "date": observation_date.date(),
            "success": False,
            "error": str(error)
        })

        print(
            "Failed:",
            observation_date.date(),
            "|",
            error
        )

Processed: 2018-07-01
Processed: 2018-07-02
Processed: 2018-07-03
Processed: 2018-07-04
Processed: 2018-07-05
Processed: 2018-07-06
Processed: 2018-07-07
Processed: 2018-07-08
Processed: 2018-07-09
Processed: 2018-07-10
Processed: 2018-07-11
Processed: 2018-07-12
Processed: 2018-07-13
Processed: 2018-07-14
Processed: 2018-07-15
Processed: 2018-07-16
Processed: 2018-07-17
Processed: 2018-07-18
Processed: 2018-07-19
Processed: 2018-07-20
Processed: 2018-07-21
Processed: 2018-07-22
Processed: 2018-07-23
Processed: 2018-07-24
Processed: 2018-07-25
Processed: 2018-07-26
Processed: 2018-07-27
Processed: 2018-07-28
Processed: 2018-07-29
Processed: 2018-07-30
Processed: 2018-07-31


In [6]:
processing_df = pd.DataFrame(
    processing_rows
)

print(
    "Successful dates:",
    int(processing_df["success"].sum()),
    "/ 31"
)

display(
    processing_df[
        ~processing_df["success"]
    ]
)

Successful dates: 31 / 31


,date,success


In [7]:
daily_files = sorted(
    DAILY_FOLDER.glob(
        "predictors_*.nc"
    )
)

if len(daily_files) != 31:
    raise ValueError(
        f"Expected 31 files, found {len(daily_files)}"
    )


daily_datasets = []

for file in daily_files:
    with xr.open_dataset(file) as ds:
        daily_datasets.append(
            ds.load()
        )


predictor_ds = xr.concat(
    daily_datasets,
    dim="date"
)

predictor_ds = predictor_ds.sortby(
    "date"
)

predictor_ds.attrs = {
    "title": "July 2018 GEFS atmospheric predictors",
    "forecast_hour": 36,
    "source": "NOAA GEFSv12 reforecast c00"
}

print(predictor_ds)

<xarray.Dataset> Size: 10MB
Dimensions:         (date: 31, latitude: 129, longitude: 121)
Coordinates:
  * date            (date) datetime64[ns] 248B 2018-07-01 ... 2018-07-31
  * latitude        (latitude) float64 1kB 6.0 6.25 6.5 6.75 ... 37.5 37.75 38.0
  * longitude       (longitude) float64 968B 68.0 68.25 68.5 ... 97.5 97.75 98.0
Data variables:
    mslp            (date, latitude, longitude) float32 2MB 1.008e+03 ... 1.0...
    u850            (date, latitude, longitude) float32 2MB 9.131 ... -7.654
    v850            (date, latitude, longitude) float32 2MB 0.6757 ... -1.168
    q850            (date, latitude, longitude) float32 2MB 10.48 ... 21.09
    wind_speed_850  (date, latitude, longitude) float32 2MB 9.156 9.58 ... 7.742
Attributes:
    title:          July 2018 GEFS atmospheric predictors
    forecast_hour:  36
    source:         NOAA GEFSv12 reforecast c00


In [8]:
quality_rows = []


for variable in predictor_ds.data_vars:
    field = predictor_ds[variable]

    quality_rows.append({
        "variable": variable,
        "minimum": float(
            field.min(skipna=True)
        ),
        "mean": float(
            field.mean(skipna=True)
        ),
        "maximum": float(
            field.max(skipna=True)
        ),
        "missing": int(
            field.isnull().sum()
        ),
        "units": field.attrs.get("units")
    })


quality_df = pd.DataFrame(
    quality_rows
)

quality_df

,variable,minimum,mean,maximum,missing,units
0,mslp,981.867371,1001.772522,1020.659241,0,hPa
1,u850,-20.387548,6.750691,30.212452,0,m/s
2,v850,-16.540440,1.012279,28.503155,0,m/s
3,q850,1.570100,14.202491,29.710001,0,g/kg
4,wind_speed_850,0.007542,8.977841,35.786854,0,m/s


In [9]:
encoding = {
    variable: {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32"
    }
    for variable in predictor_ds.data_vars
}

predictor_ds.to_netcdf(
    PREDICTOR_OUTPUT,
    mode="w",
    engine="netcdf4",
    encoding=encoding
)

print("Saved:", PREDICTOR_OUTPUT.exists())
print("Location:", PREDICTOR_OUTPUT)

Saved: True
Location: Z:\Projects\monsoon-postprocessing\data\processed\july2018_predictors.nc


In [10]:
model_ds = xr.merge(
    [
        rainfall_ds,
        predictor_ds
    ],
    join="exact"
)

model_ds.attrs = {
    "title": "July 2018 regime-aware model dataset",
    "rainfall_forecast": "GEFS +24 to +48 hours",
    "atmospheric_predictors": "GEFS forecast hour 36",
    "observation": "NASA GPM IMERG Final"
}

model_ds.to_netcdf(
    MODEL_DATASET_OUTPUT,
    mode="w",
    engine="netcdf4"
)

print("Saved:", MODEL_DATASET_OUTPUT.exists())
print("Location:", MODEL_DATASET_OUTPUT)
print(model_ds)

Saved: True
Location: Z:\Projects\monsoon-postprocessing\data\processed\july2018_model_dataset.nc
<xarray.Dataset> Size: 15MB
Dimensions:         (date: 31, latitude: 129, longitude: 121)
Coordinates:
  * date            (date) datetime64[ns] 248B 2018-07-01 ... 2018-07-31
  * latitude        (latitude) float64 1kB 6.0 6.25 6.5 6.75 ... 37.5 37.75 38.0
  * longitude       (longitude) float64 968B 68.0 68.25 68.5 ... 97.5 97.75 98.0
Data variables:
    gefs_rainfall   (date, latitude, longitude) float32 2MB 1.91 1.21 ... 1.35
    imerg_rainfall  (date, latitude, longitude) float32 2MB nan nan ... nan nan
    forecast_error  (date, latitude, longitude) float32 2MB nan nan ... nan nan
    mslp            (date, latitude, longitude) float32 2MB 1.008e+03 ... 1.0...
    u850            (date, latitude, longitude) float32 2MB 9.131 ... -7.654
    v850            (date, latitude, longitude) float32 2MB 0.6757 ... -1.168
    q850            (date, latitude, longitude) float32 2MB 10.48 ... 21.